# Dendrite Metrics Analysis

Расчет метрик для одного набора дендритов без сравнения Ab/Wt. Укажите папку с mesh-данными и папку для сохранения результатов.


In [ ]:
from pathlib import Path

import pandas as pd

from dendrite_analysis import (
    Dendrite,
    output_path,
    reset_saved_data,
    save_all_dendr_metric_dict,
    set_output_dir,
)
from notebook_widgets import SpineMeshDataset


In [ ]:
# Папка с mesh-данными.
# Для dataset_format="labid": можно указать папку одного дендрита с spine_*.off и surface_mesh.off,
# либо папку, внутри которой лежат подпапки отдельных дендритов.
# Для dataset_format="microns": укажите общую папку с нейронами, папку одного нейрона,
# папку limb_* или папку branch_*.
mesh_folder = "example_dendrite"
dataset_format = "labid"  # "labid" или "microns"

# Куда сохранять dendr_metrics.json, spine_metrics.json, grouping/cluster/graph json и итоговый CSV.
output_folder = "output_dendrite_metrics"

spine_file_pattern = "**/spine_*.off"
microns_spine_file_pattern = "*.off"
load_attachment_centers = False

# Если True, для каждого шипика будет выведен лог выбора точки крепления и два 3D-графика:
# 1) меш шипика + точка крепления; 2) меш дендрита + меш шипика + точка крепления.
# На больших датасетах это очень много вывода; при необходимости задайте attachment_debug_limit.
debug_attachment_points = False
attachment_debug_limit = None

calculate_grouping_metrics = True
calculate_cluster_metrics = True
calculate_graph_metrics = True
save_summary_csv = True


In [ ]:
mesh_root = Path(mesh_folder)
if not mesh_root.exists():
    raise FileNotFoundError(f"Папка с mesh-данными не найдена: {mesh_root}")

def find_labid_dataset_paths(mesh_root: Path):
    root_spine_files = list(mesh_root.glob(spine_file_pattern))
    if root_spine_files:
        return [mesh_root]
    return [path for path in sorted(mesh_root.iterdir()) if path.is_dir()]


def find_microns_branch_paths(mesh_root: Path):
    def is_valid_branch(path: Path) -> bool:
        return (
            path.is_dir()
            and path.name.startswith("branch_")
            and (path / "branch_mesh.off").exists()
            and (path / "spines").is_dir()
        )

    if is_valid_branch(mesh_root):
        return [mesh_root]
    return sorted(path for path in mesh_root.glob("**/branch_*") if is_valid_branch(path))


dataset_format = dataset_format.lower()
if dataset_format == "labid":
    candidate_paths = find_labid_dataset_paths(mesh_root)
    active_spine_file_pattern = spine_file_pattern
elif dataset_format == "microns":
    candidate_paths = find_microns_branch_paths(mesh_root)
    active_spine_file_pattern = microns_spine_file_pattern
else:
    raise ValueError(f"Неизвестный dataset_format={dataset_format!r}; ожидается 'labid' или 'microns'")

datasets = []
for dataset_path in candidate_paths:
    if dataset_format == "labid":
        spine_files = list(dataset_path.glob(active_spine_file_pattern))
        if not spine_files:
            continue

    spine_dataset = SpineMeshDataset().load(
        str(dataset_path),
        spine_file_pattern=active_spine_file_pattern,
        load_attachment_centers=load_attachment_centers,
        dataset_format=dataset_format,
        debug_attachment_points=debug_attachment_points,
        attachment_debug_limit=attachment_debug_limit,
    )
    if spine_dataset.spine_meshes and spine_dataset.dendrite_meshes:
        datasets.append(spine_dataset)

if not datasets:
    raise ValueError(f"В {mesh_root} не найдено подходящих данных для dataset_format={dataset_format!r}")

len(datasets)


In [ ]:
set_output_dir(output_folder)
reset_saved_data()

dendrites = []

for spine_dataset in datasets:
    dendrite = Dendrite(
        "None",
        dendrite_meshes=spine_dataset.dendrite_meshes,
        spine_meshes=spine_dataset.spine_meshes,
    )
    dendrite.save_init_metrics()

    if calculate_grouping_metrics:
        dendrite.calculate_grouping_metrics()
        dendrite.save_grouping_metrics()

    if calculate_cluster_metrics:
        dendrite.calculate_cluster_metrics()
        dendrite.save_cluster_metrics()

    if calculate_graph_metrics:
        dendrite.graph_analysis()
        dendrite.save_graph_metrics()

    if save_summary_csv:
        dendrite.save_dendr_metrics_without_class_cluster()

    dendrites.append(dendrite)

if save_summary_csv:
    pd.DataFrame(save_all_dendr_metric_dict).to_csv(output_path("all_dendr_metrics.csv"), index=False)

len(dendrites)


In [ ]:
sorted(path.name for path in Path(output_folder).iterdir())
